In [1]:
from kafka import KafkaConsumer, KafkaProducer
import json
import matplotlib.pyplot as plt
import time
import numpy as np

In [2]:
# Initialize data storage for plotting
timestamps = []
water_temperatures = []
ph_levels = []
turbidities = []
dissolved_oxygen_levels = []

In [3]:
# Kafka configuration
def initialize_consumer():
    kafka_topic = "water_quality"
    kafka_bootstrap_servers = ["localhost:9092"]

    # Create Kafka consumer
    consumer = KafkaConsumer(
        kafka_topic,
        bootstrap_servers=kafka_bootstrap_servers,
        value_deserializer=lambda m: json.loads(m.decode('utf-8')),
        auto_offset_reset='latest',
        enable_auto_commit=True
    )
    return consumer

In [4]:
# Receive all published messages and update plot
def update_plot(consumer):
    try:
        for message in consumer:
            # Parse the message
            sensor_data = message.value
            print(f"Received: {sensor_data}")

            # Update data storage
            timestamps.append(sensor_data['timestamp'])
            water_temperatures.append(sensor_data['water_temperature'])
            ph_levels.append(sensor_data['ph_level'])
            turbidities.append(sensor_data['turbidity'])
            dissolved_oxygen_levels.append(sensor_data['dissolved_oxygen'])

            # Keep only the last 100 entries for plotting
            if len(timestamps) > 100:
                timestamps.pop(0)
                water_temperatures.pop(0)
                ph_levels.pop(0)
                turbidities.pop(0)
                dissolved_oxygen_levels.pop(0)

            # Clear the current axes and redraw the plots
            plt.figure(figsize=(10, 8))

            plt.subplot(2, 2, 1)
            plt.plot(timestamps, water_temperatures, label="Water Temperature", color="blue")
            plt.title("Water Temperature")
            plt.ylabel("°C")

            plt.subplot(2, 2, 2)
            plt.plot(timestamps, ph_levels, label="pH Level", color="green")
            plt.title("pH Level")
            plt.ylabel("pH")

            plt.subplot(2, 2, 3)
            plt.plot(timestamps, turbidities, label="Turbidity", color="orange")
            plt.title("Turbidity")
            plt.ylabel("NTU")

            plt.subplot(2, 2, 4)
            plt.plot(timestamps, dissolved_oxygen_levels, label="Dissolved Oxygen", color="red")
            plt.title("Dissolved Oxygen")
            plt.ylabel("mg/L")

            plt.tight_layout()

            # Save the plot as an image
            plt.savefig(f"water_quality_plot.png")
            plt.close()

            break  # Process one message at a time
    except KeyboardInterrupt:
        print("Stopped consuming messages.")
        consumer.close()

In [ ]:
consumer = initialize_consumer()
print("Subscribed to Kafka topic 'water_quality'.")

try:
    while True:
        update_plot(consumer)
except KeyboardInterrupt:
    print("Stopped visualization.")
    consumer.close()

Subscribed to Kafka topic 'water_quality'.
Received: {'timestamp': 1772326788, 'water_temperature': 32.638654426100906, 'ph_level': 8.708669829158971, 'turbidity': 20.01, 'dissolved_oxygen': 11.74}
Received: {'timestamp': 1772326789, 'water_temperature': 31.261623551945583, 'ph_level': 8.63301762948835, 'turbidity': 9.27, 'dissolved_oxygen': 10.53}
Received: {'timestamp': 1772326790, 'water_temperature': 32.35111671136743, 'ph_level': 7.934559502972814, 'turbidity': 47.52, 'dissolved_oxygen': 7.12}
Received: {'timestamp': 1772326791, 'water_temperature': 32.38497762359722, 'ph_level': 8.595685299527815, 'turbidity': 15.48, 'dissolved_oxygen': 9.82}
Received: {'timestamp': 1772326792, 'water_temperature': 31.032320814657385, 'ph_level': 8.256232865487728, 'turbidity': 29.13, 'dissolved_oxygen': 11.76}
Received: {'timestamp': 1772326793, 'water_temperature': 31.569187609788166, 'ph_level': 8.809098956407594, 'turbidity': 20.86, 'dissolved_oxygen': 7.55}
Received: {'timestamp': 1772326794

In [5]:
OUTPUT_TOPIC = "aeration_predictions"
FEATURES = ['water_temperature', 'ph_level', 'turbidity', 'dissolved_oxygen']
INPUT_TOPIC = "water_quality"

In [6]:
import joblib
model = joblib.load('aeration_classifier.joblib')

/usr/local/lib/python3.10/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator DecisionTreeClassifier from version 1.4.0 when using version 1.6.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator RandomForestClassifier from version 1.4.0 when using version 1.6.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [7]:
# Kafka configuration
def initialize_producer():
    kafka_topic = "water_quality"
    kafka_bootstrap_servers = ["localhost:9092"]

    # Create Kafka consumer
    inference_producer = KafkaProducer(
        bootstrap_servers = kafka_bootstrap_servers,
        key_serializer    = lambda k: k.encode('utf-8'),
        value_serializer  = lambda v: json.dumps(v).encode('utf-8'),
    )
    return inference_producer
def run_inference(data: dict) -> dict:
    """
    Run RandomForest inference on one sensor reading.
    Returns a dict with sensor_id, timestamp, prediction (0/1),
    proba_no_aeration and proba_aeration from predict_proba.
    """
    X = np.array([[data[f] for f in FEATURES]])

    probas     = model.predict_proba(X)[0]    # shape (2,)
    prediction = int(model.predict(X)[0])     # 0 or 1

    class_to_idx = {int(c): i for i, c in enumerate(model.classes_)}
    p_no_aer     = float(probas[class_to_idx[0]])
    p_aer        = float(probas[class_to_idx[1]])

    return {
        'sensor_id':         data.get('sensor_id', 'unknown'),
        'timestamp':         data.get('timestamp', int(time.time())),
        'prediction':        prediction,          # 0 = no aeration, 1 = aeration
        'proba_no_aeration': round(p_no_aer, 4),  # P(class 0)
        'proba_aeration':    round(p_aer, 4),     # P(class 1)
    }

In [ ]:
consumer = initialize_consumer()
producer = initialize_producer()
print("Subscribed to Kafka topic 'water_quality'.")

try:
    while True:
        for message in consumer:
            sensor_data = message.value
            result = run_inference(sensor_data)        
            print(f"Sensor {result['sensor_id']} | Pred: {result['prediction']} | P(0): {result['proba_no_aeration']} | P(1): {result['proba_aeration']}") 
            producer.send(OUTPUT_TOPIC, key=str(result['sensor_id']), value=result)
except KeyboardInterrupt:
    print("**Deteniendo procesador ML.**")
    consumer.close()
    producer.close()

Subscribed to Kafka topic 'water_quality'.
Sensor 3 | Pred: 0 | P(0): 1.0 | P(1): 0.0
Sensor 0 | Pred: 0 | P(0): 1.0 | P(1): 0.0
Sensor 1 | Pred: 0 | P(0): 1.0 | P(1): 0.0
Sensor 2 | Pred: 0 | P(0): 1.0 | P(1): 0.0
Sensor 3 | Pred: 1 | P(0): 0.1 | P(1): 0.9
Sensor 0 | Pred: 0 | P(0): 1.0 | P(1): 0.0
Sensor 1 | Pred: 0 | P(0): 1.0 | P(1): 0.0
Sensor 2 | Pred: 1 | P(0): 0.0 | P(1): 1.0
Sensor 3 | Pred: 0 | P(0): 1.0 | P(1): 0.0
Sensor 0 | Pred: 1 | P(0): 0.1 | P(1): 0.9
Sensor 1 | Pred: 0 | P(0): 1.0 | P(1): 0.0
Sensor 2 | Pred: 0 | P(0): 1.0 | P(1): 0.0
Sensor 3 | Pred: 1 | P(0): 0.0 | P(1): 1.0
Sensor 0 | Pred: 0 | P(0): 1.0 | P(1): 0.0
Sensor 1 | Pred: 1 | P(0): 0.0 | P(1): 1.0
Sensor 2 | Pred: 0 | P(0): 1.0 | P(1): 0.0
Sensor 3 | Pred: 0 | P(0): 1.0 | P(1): 0.0
Sensor 0 | Pred: 1 | P(0): 0.0 | P(1): 1.0
Sensor 1 | Pred: 1 | P(0): 0.0 | P(1): 1.0
Sensor 2 | Pred: 0 | P(0): 1.0 | P(1): 0.0
Sensor 3 | Pred: 0 | P(0): 0.8 | P(1): 0.2
Sensor 0 | Pred: 1 | P(0): 0.4 | P(1): 0.6
Sensor 1 | 